In [ ]:
import clip
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data.dataloader import DataLoader
from tqdm import tqdm

from ca_sae.sae.batch_topk import BatchTopKSAE
from ca_sae.sae.softsae import SoftSAE
from ca_sae.sae.ca_sae import ClasSAE

In [ ]:
clip_model, clip_preprocess = clip.load("ViT-L/14")

In [ ]:
def transform(batch):
    batch["pixel_values"] = [
        clip_preprocess(image.convert("RGB"))
        for image in batch["image"]
    ]
    del batch['image']
    return batch



imagenet100 = load_dataset(
    "clane9/imagenet-100", cache_dir="../hf_cache", split="validation"
)
imagenet100.set_transform(transform)

In [ ]:
dataloader = DataLoader(imagenet100, batch_size=64)

In [ ]:
# sae = SoftSAE.from_pretrained("../checkpoints/test/rework_v1_basic_l0/ae.pt").cuda()
# sae = SoftSAE.from_pretrained("../checkpoints/test/SoftSAE_4096_64/ae.pt").cuda()
# sae = SoftSAE.from_pretrained("../checkpoints/test/rework_v1_gradient_trick/ae.pt").cuda()
# sae = ClasSAE.from_pretrained("../checkpoints/test/ca_sae_v1/ae.pt", rho=5).cuda()
# sae = ClasSAE.from_pretrained("../checkpoints/test/ca_sae_v2").cuda()
sae = ClasSAE.from_pretrained("../checkpoints/test/ca_sae_tight_l0_agr/").cuda()
# sae = BatchTopKSAE.from_pretrained("../checkpoints/test/BatchTopKSAESecond/ae.pt").cuda()
sae = sae.eval()

In [ ]:
clip_embeddings= []
sae_embeddings = []
labels = []

with torch.inference_mode():
    for batch in tqdm(dataloader):
        images = batch['pixel_values'].cuda()
        labels_batch = batch['label']

        clip_out = clip_model.encode_image(images)
        sae_out = sae.encode(clip_out)

        clip_embeddings.append(clip_out)
        sae_embeddings.append(sae_out)
        labels.append(labels_batch)
        

In [ ]:
clip_embeddings = torch.cat(clip_embeddings)
sae_embeddings = torch.cat(sae_embeddings)
labels = torch.cat(labels)

# K Distribution

In [ ]:
ks = (sae_embeddings !=  0).float().sum(dim=-1).cpu().numpy()

plt.hist(ks, bins=50, density=True)

# Feature examinationn

In [ ]:
print("Dead features:",   torch.all(sae_embeddings == 0, dim=0).sum().item())

In [ ]:
preview_ds = load_dataset(
    "clane9/imagenet-100", cache_dir="../hf_cache", split="validation"
)

In [ ]:
row, col = torch.unravel_index(sae_embeddings.argmax(), sae_embeddings.shape)

examples_ = sae_embeddings[:, 1].topk(10)
print(examples_.values)
examples  = examples_.indices[examples_.values.nonzero()]
examples
col

In [ ]:
sae_embeddings.mean(dim=0).topk(10).indices

In [ ]:
for e in examples:
    plt.axis('off')
    plt.imshow(preview_ds[e.item()]['image'])
    plt.show()